# 65. ACE Magnetic Fields Experiment (MAG) — Implementation / ACE 자기장 실험 구현

**Paper**: Smith, C. W., L'Heureux, J., Ness, N. F., Acuña, M. H., Burlaga, L. F., & Scheifele, J. (1998). "The ACE Magnetic Fields Experiment." *Space Science Reviews* **86**, 613–632. DOI: [10.1023/A:1005092216668](https://doi.org/10.1023/A:1005092216668)

This notebook reproduces four key concepts of the ACE/MAG instrument and its space-weather operations:
1. Vector measurement on a spinning rotor and the despin transformation that recovers the inertial-frame IMF.
2. Coordinate transformation between **GSE** and **GSM** frames.
3. Detection of an **ICME magnetic-cloud signature** by recognising a smooth, large-amplitude rotation of $B_z$.
4. The **real-time alert criterion** based on a sustained $B_z < B_\text{th}$ window — a simplified version of NOAA SWPC's RTSW pipeline.

이 노트북은 ACE/MAG 기기와 우주 기상 운용의 네 가지 핵심 개념을 재현한다.
1. 회전(스핀) 위성 위에서의 벡터 측정과, 관성계 IMF를 복원하는 디스핀 변환.
2. **GSE ↔ GSM** 좌표계 변환.
3. $B_z$의 부드럽고 큰 진폭 회전을 감지하여 **ICME 자기 구름 시그니처**를 검출.
4. 지속된 $B_z < B_\text{th}$ 윈도우에 기반한 **실시간 경보 기준** — NOAA SWPC RTSW 파이프라인의 단순화 버전.

All synthetic data are generated in this notebook (no external file dependencies); kernel **`study-with-ai`**.
모든 합성 데이터는 본 노트북 내에서 생성한다 (외부 파일 의존 없음); 커널 **`study-with-ai`**.

In [ ]:
"""Standard imports and matplotlib defaults for the ACE/MAG implementation notebook."""
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import welch
from dataclasses import dataclass
from typing import Tuple

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

# Reproducibility / 재현성을 위한 고정 시드
RNG = np.random.default_rng(seed=19970825)  # ACE launch date
print('NumPy', np.__version__)

## Part 1: Spinning-Rotor Vector Measurement & Despin / 스핀 위성의 벡터 측정과 디스핀

ACE rotates at about **5 RPM** (period $T_s = 12$ s, angular frequency $\omega_s = \pi/6$ rad/s ≈ 0.524 rad/s) with its spin axis nominally along $+\hat{Z}_\text{GSE}$ (anti-sunward). The two MAG fluxgate sensors sit at the tips of the $\pm\hat{Y}$ booms and rotate with the spacecraft. A truly stationary inertial-frame IMF $(B_x^I, B_y^I, B_z^I)$ therefore appears, in raw spacecraft coordinates, as:

$$ B_x^{S}(t) = \;\;\,\cos\phi(t)\, B_x^{I} + \sin\phi(t)\, B_y^{I}, $$
$$ B_y^{S}(t) = -\sin\phi(t)\, B_x^{I} + \cos\phi(t)\, B_y^{I}, $$
$$ B_z^{S}(t) = B_z^{I}, $$

where $\phi(t) = \omega_s\, t + \phi_0$ is the spin phase taken from the sun-sensor pulse. The despin operation is the inverse rotation — multiplying $(B_x^{S}, B_y^{S})$ by $R(-\phi(t))$ — which removes the dominant spin tone at $f_s = 1/T_s \approx 0.0833$ Hz and restores the true IMF.

ACE는 약 **5 RPM** (주기 $T_s = 12$ s, 각속도 $\omega_s = \pi/6$ rad/s ≈ 0.524 rad/s)로 회전하며, 스핀축은 명목상 $+\hat{Z}_\text{GSE}$ (반-태양 방향)에 정렬된다. 두 MAG 플럭스게이트 센서는 $\pm\hat{Y}$ 붐 끝에 위치하여 위성과 함께 회전한다. 관성계에서 정지한 IMF $(B_x^I, B_y^I, B_z^I)$는 위성 좌표계에서 위와 같이 보이며, 디스핀 ($R(-\phi(t))$ 곱셈)이 스핀 주파수 $f_s = 1/T_s \approx 0.0833$ Hz의 강한 톤을 제거하고 진성 IMF를 복원한다.

**Why do this on board?** TI320C10 DSP performs despin **before** the FFT so that the 32 log-spaced spectral channels (0–12 Hz) reflect IMF turbulence rather than spin-tone harmonics (notes §4.3, p. 627).

**왜 온보드에서 수행하는가?** TI320C10 DSP는 FFT **이전**에 디스핀을 수행하여 32 로그 간격 스펙트럼 채널 (0–12 Hz)이 스핀 톤 고조파 대신 진성 IMF 난류를 반영하도록 한다 (노트 §4.3, p. 627).

In [ ]:
"""Simulate ACE/MAG raw rotor data and despin to inertial frame.

We construct a synthetic IMF time series in the inertial (GSE) frame, project it into the
rotating spacecraft frame using the spin-phase rotation R(phi), then recover the inertial
field by applying R(-phi). The Welch PSD reveals the spin tone at f_s = 1/12 Hz before despin
and its disappearance afterward.
"""

# ----- Instrument constants from Smith et al. 1998 -----
FS = 24.0          # vector samples per second (snapshot rate)
T_SPIN = 12.0      # spin period (s) -> 5 RPM
OMEGA_S = 2.0 * np.pi / T_SPIN  # spin angular frequency (rad/s)
F_SPIN = 1.0 / T_SPIN           # spin tone frequency (Hz)
NOISE_RMS = 0.006  # nT RMS, 0-10 Hz (Smith et al. 1998 spec)

# ----- Build a synthetic 5-minute inertial IMF (GSE) -----
duration_s = 300.0
t = np.arange(0.0, duration_s, 1.0 / FS)
n_samples = t.size

# Quiet-time IMF: Parker spiral ~ 45 deg in ecliptic, |B| ~ 5 nT
B_inertial = np.zeros((3, n_samples))
B_inertial[0] = 3.5 + 0.5 * np.sin(2 * np.pi * 0.05 * t)   # Bx_GSE slow variation
B_inertial[1] = -3.5 + 0.5 * np.cos(2 * np.pi * 0.07 * t)  # By_GSE slow variation
B_inertial[2] = 1.0 + 0.3 * np.sin(2 * np.pi * 0.10 * t)   # Bz_GSE small

# Add white noise at instrument level (0.006 nT RMS over 0-10 Hz roughly)
B_inertial += NOISE_RMS * RNG.standard_normal(B_inertial.shape)


def rotation_matrix_z(phi: np.ndarray) -> np.ndarray:
    """Return a stack of Z-axis rotation matrices, one per time sample.

    Args:
        phi: Spin phase in radians, shape (N,).

    Returns:
        Rotation matrix array of shape (N, 3, 3) such that
        v_rotated = R(phi) @ v_inertial element-wise per sample.
    """
    cos_phi = np.cos(phi)
    sin_phi = np.sin(phi)
    n = phi.size
    rot = np.zeros((n, 3, 3))
    rot[:, 0, 0] = cos_phi
    rot[:, 0, 1] = sin_phi
    rot[:, 1, 0] = -sin_phi
    rot[:, 1, 1] = cos_phi
    rot[:, 2, 2] = 1.0
    return rot


def project_to_spacecraft(B_inert: np.ndarray, phi: np.ndarray) -> np.ndarray:
    """Rotate inertial-frame field into spacecraft (rotating) coordinates.

    Args:
        B_inert: Inertial-frame field, shape (3, N).
        phi: Spin phase per sample, shape (N,).

    Returns:
        Field in spacecraft (rotating) frame, shape (3, N).
    """
    rot = rotation_matrix_z(phi)                  # (N, 3, 3)
    out = np.einsum('nij,jn->in', rot, B_inert)
    return out


def despin(B_sc: np.ndarray, phi: np.ndarray) -> np.ndarray:
    """Recover the inertial-frame field by inverse rotation.

    Args:
        B_sc: Spacecraft (rotating) frame field, shape (3, N).
        phi: Spin phase per sample, shape (N,).

    Returns:
        Despun (inertial-frame) field, shape (3, N).
    """
    rot_inv = rotation_matrix_z(-phi)             # R(-phi)
    return np.einsum('nij,jn->in', rot_inv, B_sc)


# Apply forward (project) then inverse (despin)
phi_t = OMEGA_S * t + 0.4  # initial spin phase phi_0 = 0.4 rad
B_spacecraft = project_to_spacecraft(B_inertial, phi_t)
B_despun = despin(B_spacecraft, phi_t)

# Verify round-trip accuracy
round_trip_err = np.max(np.abs(B_despun - B_inertial))
print(f'Spin period           = {T_SPIN:.1f} s')
print(f'Spin tone frequency   = {F_SPIN:.4f} Hz')
print(f'Round-trip max error  = {round_trip_err:.2e} nT (numerical only)')

In [ ]:
"""Visualise raw spacecraft-frame Bx,By vs despun inertial Bx,By and the spin tone in PSD."""
fig, axes = plt.subplots(2, 2, figsize=(12, 7))

# Top-left: raw spacecraft-frame Bx, By show 0.0833 Hz spin modulation
axes[0, 0].plot(t, B_spacecraft[0], label=r'$B_x^{S}$ (raw, spinning)', lw=0.8)
axes[0, 0].plot(t, B_spacecraft[1], label=r'$B_y^{S}$ (raw, spinning)', lw=0.8)
axes[0, 0].set_xlim(0, 60)
axes[0, 0].set_xlabel('Time (s)')
axes[0, 0].set_ylabel('B (nT)')
axes[0, 0].set_title('Spacecraft frame (raw) / 위성 좌표계 (원시)')
axes[0, 0].legend(loc='upper right', fontsize=9)

# Top-right: despun (inertial-frame) Bx, By recover the true slow IMF
axes[0, 1].plot(t, B_despun[0], label=r'$B_x^{I}$ despun', lw=0.8)
axes[0, 1].plot(t, B_despun[1], label=r'$B_y^{I}$ despun', lw=0.8)
axes[0, 1].set_xlim(0, 60)
axes[0, 1].set_xlabel('Time (s)')
axes[0, 1].set_ylabel('B (nT)')
axes[0, 1].set_title('Inertial frame (despun) / 관성계 (디스핀 후)')
axes[0, 1].legend(loc='upper right', fontsize=9)

# Bottom: PSD of Bx in both frames
f_raw, psd_raw = welch(B_spacecraft[0], fs=FS, nperseg=512)
f_dsp, psd_dsp = welch(B_despun[0],     fs=FS, nperseg=512)
axes[1, 0].loglog(f_raw, psd_raw, label='raw spacecraft frame')
axes[1, 0].loglog(f_dsp, psd_dsp, label='despun inertial frame', alpha=0.85)
axes[1, 0].axvline(F_SPIN, color='red', ls='--', lw=1, label=f'$f_s = {F_SPIN:.4f}$ Hz')
axes[1, 0].set_xlabel('Frequency (Hz)')
axes[1, 0].set_ylabel(r'PSD (nT$^2$/Hz)')
axes[1, 0].set_title(r'$B_x$ PSD: spin tone vanishes after despin / 디스핀 후 스핀 톤 소멸')
axes[1, 0].legend(fontsize=9)

# Bottom-right: Bz unchanged (parallel to spin axis)
axes[1, 1].plot(t, B_spacecraft[2], lw=0.8, label='raw $B_z^{S}$')
axes[1, 1].plot(t, B_despun[2], lw=0.8, ls='--', label='despun $B_z^{I}$')
axes[1, 1].set_xlabel('Time (s)')
axes[1, 1].set_ylabel(r'$B_z$ (nT)')
axes[1, 1].set_title(r'$B_z$ unaffected by spin (parallel to spin axis) / 스핀축 평행이라 영향 없음')
axes[1, 1].legend(fontsize=9)

plt.tight_layout()
plt.show()

## Part 2: GSE ↔ GSM Coordinate Transformation / GSE ↔ GSM 좌표 변환

ACE/MAG Level-2 data are published in **GSE** (Geocentric Solar Ecliptic) but space-weather alerts use **GSM** (Geocentric Solar Magnetospheric), because magnetospheric reconnection responds to $B_z^\text{GSM}$ — the field component perpendicular to the magnetopause aligned with Earth's dipole tilt.

Both frames share $\hat{X} = \hat{X}_\text{GSE} = \hat{X}_\text{GSM}$ (Sun → Earth). GSM is then obtained from GSE by a rotation about $\hat{X}$ through the **dipole-tilt** angle $\psi(t)$, which annually oscillates roughly between $\pm 35^\circ$ as Earth's magnetic dipole projects onto the ecliptic plane:

$$\begin{pmatrix} X \\ Y \\ Z \end{pmatrix}_\text{GSM} = \begin{pmatrix} 1 & 0 & 0 \\ 0 & \cos\psi & \sin\psi \\ 0 & -\sin\psi & \cos\psi \end{pmatrix} \begin{pmatrix} X \\ Y \\ Z \end{pmatrix}_\text{GSE}$$

ACE/MAG Level-2 데이터는 **GSE** (지심 태양 황도) 좌표계로 공개되지만, 우주 기상 경보는 **GSM** (지심 태양 자기권) 좌표계의 $B_z^\text{GSM}$를 사용한다. 자기권 재결합은 황도면이 아닌 지구 쌍극자 축의 기울기에 정렬된 자기권계면 수직 성분에 반응하기 때문이다. 두 좌표계는 $\hat{X}$ 축 (태양 방향)을 공유하며, $\hat{X}$ 축 주변에서 **쌍극자 기울기** 각 $\psi(t)$ 만큼의 회전으로 GSM을 얻는다. $\psi$는 1년 동안 대략 $\pm 35^\circ$ 사이에서 진동한다.

Below we use a simplified sinusoidal $\psi(t)$ as a stand-in for the full Hapgood (1992) transformation.
아래에서는 Hapgood (1992) 정식 변환의 단순화 버전으로 정현파 $\psi(t)$를 사용한다.

In [ ]:
"""Rotate GSE field components into GSM via dipole-tilt rotation about X-axis.

We use a simplified annual-only dipole-tilt model: psi(t) ≈ psi_max * sin(2*pi*(doy - 80)/365.25).
Reality (Hapgood 1992) also includes a diurnal modulation; the form below is sufficient to illustrate
how a fixed inertial Bz_GSE projects onto a time-varying Bz_GSM seen by SWPC.
"""

PSI_MAX_DEG = 35.0  # peak annual dipole-tilt amplitude (deg) — approximate


def dipole_tilt_deg(day_of_year: np.ndarray) -> np.ndarray:
    """Approximate annual dipole-tilt angle psi (degrees).

    Args:
        day_of_year: Day of year (1-365.25), float array.

    Returns:
        Tilt angle psi in degrees; positive when Earth's north magnetic pole
        points toward the Sun.
    """
    return PSI_MAX_DEG * np.sin(2.0 * np.pi * (day_of_year - 80.0) / 365.25)


def gse_to_gsm(B_gse: np.ndarray, psi_deg: float) -> np.ndarray:
    """Rotate a 3-vector field from GSE into GSM about the shared X-axis.

    Args:
        B_gse: Field in GSE coordinates, shape (3,) or (3, N).
        psi_deg: Dipole-tilt angle in degrees (scalar).

    Returns:
        Field rotated into GSM coordinates, same shape as input.
    """
    psi = np.deg2rad(psi_deg)
    rot = np.array([
        [1.0,         0.0,          0.0],
        [0.0,  np.cos(psi),  np.sin(psi)],
        [0.0, -np.sin(psi),  np.cos(psi)],
    ])
    return rot @ B_gse


# Demonstrate: a constant southward IMF in GSE and how its GSM projection varies seasonally
B_gse_const = np.array([0.0, 0.0, -10.0])  # purely southward GSE Bz = -10 nT
doys = np.linspace(1, 365.25, 366)
psis = dipole_tilt_deg(doys)
B_gsm_z_year = np.array([gse_to_gsm(B_gse_const, p)[2] for p in psis])

fig, ax = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
ax[0].plot(doys, psis, color='C2')
ax[0].set_ylabel(r'Dipole tilt $\psi$ (deg)')
ax[0].set_title(r'Annual dipole tilt and resulting $B_z^\mathrm{GSM}$ for $B_z^\mathrm{GSE} = -10$ nT')
ax[0].axhline(0, color='k', lw=0.5)
ax[1].plot(doys, B_gsm_z_year, color='C3')
ax[1].axhline(-10.0, color='k', ls='--', lw=0.7, label=r'$B_z^\mathrm{GSE} = -10$ nT')
ax[1].set_xlabel('Day of year / 1년 중 일자')
ax[1].set_ylabel(r'$B_z^\mathrm{GSM}$ (nT)')
ax[1].legend(fontsize=9)
plt.tight_layout()
plt.show()

print(f'Range of Bz_GSM over the year for fixed Bz_GSE=-10 nT: '
      f'[{B_gsm_z_year.min():+.2f}, {B_gsm_z_year.max():+.2f}] nT')

## Part 3: ICME Magnetic-Cloud Detection / ICME 자기 구름 검출

Burlaga (1995) defined a **magnetic cloud** as an interplanetary structure with:
1. Enhanced magnetic field magnitude $|B|$ relative to ambient solar wind (typically 2–3×).
2. **Smooth, large-amplitude rotation** of the field direction over $\sim 10$–$30$ hours.
3. Low proton temperature ($T_p < 0.5\, T_\text{expected}$).

Magnetic clouds are a well-organised subset of ICMEs — those with intact flux-rope topology. ACE/MAG's smooth, high-cadence vector field is ideal for recognising criterion (2): we model the cloud as a force-free, cylindrically symmetric flux rope (Lundquist 1950; Burlaga 1988) and detect it via

* a sustained jump in $|B|$ above an ambient threshold,
* a smooth, monotonic rotation of $B_z^\text{GSM}$ from one polarity to the other (low jitter / high autocorrelation).

Burlaga (1995)는 **자기 구름**을 다음 조건을 모두 만족하는 행성간 구조로 정의했다.
1. 주변 태양풍 대비 증가된 자기장 크기 $|B|$ (보통 2–3배).
2. 약 $10$–$30$ 시간에 걸친 **부드럽고 큰 진폭의** 자기장 방향 회전.
3. 낮은 양성자 온도 ($T_p < 0.5\, T_\text{expected}$).

자기 구름은 무결한 flux-rope 위상을 갖는 ICME 부분 집합이다. ACE/MAG의 매끄러운 고샘플링 벡터 자기장은 조건 (2)를 인식하는 데 이상적이다. 우리는 자기 구름을 force-free 원통 대칭 flux rope (Lundquist 1950; Burlaga 1988)로 모델링하고, (a) $|B|$의 임계값 초과 지속, (b) $B_z^\text{GSM}$의 부드러운 단조 회전으로 검출한다.

In [ ]:
"""Generate a synthetic 3-day ACE time series with an embedded magnetic cloud, then detect it.

We model the cloud as a Lundquist flux rope sampled along a straight ACE trajectory through the rope.
The detector flags intervals where (i) |B| exceeds twice the ambient median and (ii) Bz_GSM rotates
monotonically by more than DELTA_BZ over a sustained window with low high-frequency jitter.
"""

@dataclass
class CloudParams:
    """Parameters for a synthetic Lundquist flux-rope encounter."""
    t_start_h: float       # cloud start time (hours from sequence start)
    duration_h: float      # cloud duration (hours)
    b0_nt: float           # peak axial field (nT)
    impact_ratio: float    # closest approach / cloud radius (0 = axis crossing)


def lundquist_flux_rope(t_h: np.ndarray, p: CloudParams) -> np.ndarray:
    """Return GSM-frame field components along an ACE trajectory through a flux rope.

    Uses the Bessel-function force-free Lundquist (1950) solution simplified to the standard
    smooth rotation Burlaga used for IMP-8 and ACE classifications.

    Args:
        t_h: Time in hours, shape (N,).
        p: CloudParams describing the encounter.

    Returns:
        Field array of shape (3, N) inside the cloud, zeros outside.
    """
    n = t_h.size
    out = np.zeros((3, n))
    inside = (t_h >= p.t_start_h) & (t_h <= p.t_start_h + p.duration_h)
    if not np.any(inside):
        return out
    # Normalised position from -1 (entry) to +1 (exit)
    s = 2.0 * (t_h[inside] - p.t_start_h) / p.duration_h - 1.0
    r = np.sqrt(s**2 + p.impact_ratio**2)              # normalised radial distance
    # Smooth rotation: Bz_GSM rotates from +B0 to -B0 (south-then-north or vice versa)
    theta = np.pi * (1.0 - r)                          # 0 at edge -> pi near axis
    bz_amp = -p.b0_nt * np.cos(np.pi * (s + 1.0) / 2.0)  # smooth s-shape (+B0 -> -B0)
    bx_amp = 0.5 * p.b0_nt * np.sin(theta)               # axial-direction component
    by_amp = 0.5 * p.b0_nt * np.sin(theta) * np.sign(s)
    out[0, inside] = bx_amp
    out[1, inside] = by_amp
    out[2, inside] = bz_amp
    return out


# ----- Build a 72-hour synthetic ACE/MAG record at 1-minute cadence -----
T_HOURS = 72.0
DT_MIN = 1.0
t_min = np.arange(0.0, T_HOURS * 60.0, DT_MIN)
t_h = t_min / 60.0

# Quiet-time ambient IMF (Parker spiral) plus turbulence (random walk)
B_ambient = np.zeros((3, t_h.size))
B_ambient[0] = 3.0 + 0.7 * np.sin(2 * np.pi * t_h / 18.0)
B_ambient[1] = -3.0 + 0.7 * np.cos(2 * np.pi * t_h / 24.0)
B_ambient[2] = 0.0 + 0.4 * np.sin(2 * np.pi * t_h / 13.0)
B_ambient += 0.5 * RNG.standard_normal(B_ambient.shape)

# Embed a magnetic cloud from t = 30 h, duration 22 h, peak |B| = 18 nT
cloud = CloudParams(t_start_h=30.0, duration_h=22.0, b0_nt=18.0, impact_ratio=0.2)
B_cloud = lundquist_flux_rope(t_h, cloud)
B_total = B_ambient + B_cloud
B_mag = np.linalg.norm(B_total, axis=0)
Bz = B_total[2]

print(f'Ambient median |B|     = {np.median(B_ambient[0]**2 + B_ambient[1]**2 + B_ambient[2]**2)**0.5:.2f} nT')
print(f'Cloud peak |B|         = {B_mag.max():.2f} nT')
print(f'Cloud Bz excursion     = [{Bz.min():+.2f}, {Bz.max():+.2f}] nT')

In [ ]:
"""Detect magnetic-cloud intervals via |B| enhancement plus smooth Bz rotation."""

def smoothness(x: np.ndarray) -> float:
    """Return a smoothness score: ratio of std(x) to std(diff(x)).

    Larger values indicate the signal varies slowly relative to its sample-to-sample jitter,
    consistent with the smooth flux-rope rotation expected inside a magnetic cloud.

    Args:
        x: Time series.

    Returns:
        Smoothness score (higher is smoother).
    """
    if x.size < 3 or np.std(np.diff(x)) == 0:
        return 0.0
    return float(np.std(x) / np.std(np.diff(x)))


def detect_magnetic_cloud(
    t_h: np.ndarray,
    bmag: np.ndarray,
    bz: np.ndarray,
    bmag_threshold: float = 8.0,
    min_duration_h: float = 8.0,
    bz_swing_min: float = 10.0,
    smooth_min: float = 5.0,
) -> list:
    """Find candidate magnetic-cloud intervals.

    A candidate must satisfy: (1) |B| > threshold for at least min_duration_h hours,
    (2) Bz peak-to-peak swing > bz_swing_min nT inside the interval,
    (3) smoothness score of Bz > smooth_min.

    Args:
        t_h: Time in hours.
        bmag: |B| series in nT.
        bz: Bz_GSM series in nT.
        bmag_threshold: |B| floor to enter cloud candidate (nT).
        min_duration_h: Minimum sustained duration of |B| enhancement (h).
        bz_swing_min: Minimum Bz peak-to-peak rotation amplitude (nT).
        smooth_min: Minimum smoothness score of Bz inside the candidate.

    Returns:
        List of (t_start, t_end, bz_swing, smooth_score) tuples for accepted candidates.
    """
    in_cloud = bmag > bmag_threshold
    candidates: list = []
    i = 0
    while i < t_h.size:
        if not in_cloud[i]:
            i += 1
            continue
        j = i
        while j < t_h.size and in_cloud[j]:
            j += 1
        dur = t_h[j - 1] - t_h[i]
        if dur >= min_duration_h:
            bz_segment = bz[i:j]
            swing = float(bz_segment.max() - bz_segment.min())
            sm = smoothness(bz_segment)
            if swing >= bz_swing_min and sm >= smooth_min:
                candidates.append((float(t_h[i]), float(t_h[j - 1]), swing, sm))
        i = j
    return candidates


found = detect_magnetic_cloud(t_h, B_mag, Bz)
for k, (t0, t1, swing, sm) in enumerate(found):
    print(f'Candidate {k+1}: t = {t0:.1f} h to {t1:.1f} h '
          f'(duration {t1-t0:.1f} h, Bz swing = {swing:.1f} nT, smoothness = {sm:.2f})')

# Plot |B| and Bz_GSM with detected interval shaded
fig, ax = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
ax[0].plot(t_h, B_mag, color='C0', lw=0.9)
ax[0].axhline(8.0, color='gray', ls='--', lw=0.7, label='|B| threshold = 8 nT')
ax[0].set_ylabel('|B| (nT)')
ax[0].legend(fontsize=9)
ax[1].plot(t_h, Bz, color='C3', lw=0.9)
ax[1].axhline(0.0, color='k', lw=0.5)
ax[1].set_xlabel('Time (h)')
ax[1].set_ylabel(r'$B_z^\mathrm{GSM}$ (nT)')
for t0, t1, _, _ in found:
    for a in ax:
        a.axvspan(t0, t1, color='gold', alpha=0.25, label='detected cloud' if a is ax[0] else None)
ax[0].set_title('Synthetic ACE/MAG: magnetic-cloud detection / 자기 구름 검출')
plt.tight_layout()
plt.show()

## Part 4: Real-Time Alert Criterion (Sustained Southward $B_z$) / 실시간 경보 기준

Gonzalez et al. (1994) showed that **sustained, strongly southward $B_z^\text{GSM}$** is the dominant driver of geomagnetic storms. NOAA SWPC's RTSW pipeline therefore raises a watch when

$$ \mathcal{A}(t) = \begin{cases} 1 & \text{if } B_z^\text{GSM}(\tau) < B_\text{th} \;\forall\,\tau \in [t-\Delta t_\text{th}, t] \\ 0 & \text{otherwise.} \end{cases} $$

Operational defaults (notes §4.4): $B_\text{th} = -10$ nT and $\Delta t_\text{th} = 15$ min for a G1–G2 watch. The advance warning is set by the L1 → Earth solar-wind propagation lag

$$ \Delta t_{L_1\to\text{Earth}} = \frac{1.5 \times 10^6\, \text{km}}{v_\text{sw}} $$

→ ~62 min at 400 km/s, ~31 min at 800 km/s.

Gonzalez et al. (1994)는 **지속된 강한 남향 $B_z^\text{GSM}$**이 지자기 폭풍의 주요 driver임을 밝혔다. NOAA SWPC RTSW 파이프라인은 위 조건을 만족하면 watch를 발령한다. 운용 기본값 (노트 §4.4): $B_\text{th} = -10$ nT, $\Delta t_\text{th} = 15$분 → G1–G2 watch. 사전 경보 시간은 L1 → 지구 태양풍 전파 지연으로 설정되며, 400 km/s에서 약 62분, 800 km/s에서 약 31분이다.

Below, we run this exact criterion over the synthetic record from Part 3 and show the lead-time NOAA would have for different solar-wind speeds.

아래에서는 Part 3의 합성 데이터에 동일 기준을 적용하고, 다양한 태양풍 속도에 대해 NOAA가 확보할 사전 경보 시간을 보여준다.

In [ ]:
"""Implement the sustained southward Bz alert and compute lead time vs. solar-wind speed."""

L1_TO_EARTH_KM = 1.5e6  # km, ACE-Earth distance at L1


def sustained_bz_alert(
    t_min: np.ndarray,
    bz: np.ndarray,
    bz_threshold: float = -10.0,
    dwell_min: float = 15.0,
) -> np.ndarray:
    """Return a boolean alert array A(t) per Gonzalez/Burton-style criterion.

    A(t) = 1 if Bz(tau) < bz_threshold for ALL tau in [t - dwell_min, t].

    Args:
        t_min: Time vector in minutes (uniform spacing assumed).
        bz: Bz_GSM time series in nT.
        bz_threshold: Threshold (nT, negative for southward).
        dwell_min: Required sustained-time window (minutes).

    Returns:
        Boolean array of the same shape as t_min.
    """
    dt = float(np.median(np.diff(t_min)))
    window_n = max(1, int(round(dwell_min / dt)))
    below = bz < bz_threshold
    # rolling min of `below` over window_n samples ending at index i
    alert = np.zeros_like(below, dtype=bool)
    cum = np.concatenate(([0], np.cumsum(below.astype(int))))
    for i in range(window_n - 1, t_min.size):
        win_count = cum[i + 1] - cum[i + 1 - window_n]
        alert[i] = win_count == window_n
    return alert


def l1_to_earth_lead_min(v_sw_km_s: float) -> float:
    """Solar-wind propagation lag from L1 to Earth in minutes.

    Args:
        v_sw_km_s: Solar-wind bulk speed (km/s).

    Returns:
        Lead time available to Earth-based forecasters in minutes.
    """
    return L1_TO_EARTH_KM / v_sw_km_s / 60.0


alert_flags = sustained_bz_alert(t_min, Bz, bz_threshold=-10.0, dwell_min=15.0)
n_alert = int(alert_flags.sum())
first_alert_idx = int(np.argmax(alert_flags)) if n_alert > 0 else -1
first_alert_time_h = t_h[first_alert_idx] if first_alert_idx >= 0 else None

print(f'Alert samples: {n_alert} / {alert_flags.size}')
if first_alert_time_h is not None:
    print(f'First alert time     = {first_alert_time_h:.2f} h after sequence start')
    for v in (350.0, 400.0, 600.0, 800.0):
        lead_min = l1_to_earth_lead_min(v)
        print(f'  Lead time at v_sw = {v:5.0f} km/s: {lead_min:5.1f} min')

# Visualise alert window
fig, ax = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
ax[0].plot(t_h, Bz, color='C3', lw=0.9, label=r'$B_z^\mathrm{GSM}$')
ax[0].axhline(-10.0, color='k', ls='--', lw=0.7, label=r'$B_\mathrm{th} = -10$ nT')
ax[0].set_ylabel('Bz (nT)')
ax[0].legend(loc='lower right', fontsize=9)
ax[1].step(t_h, alert_flags.astype(int), color='C1', where='post')
ax[1].set_yticks([0, 1])
ax[1].set_ylabel('Alert A(t)')
ax[1].set_xlabel('Time (h)')
ax[1].set_title('Sustained southward $B_z$ alert / 지속적 남향 $B_z$ 경보')
plt.tight_layout()
plt.show()

## Summary / 요약

| Concept / 개념 | This Paper / 이 논문 | Modern Equivalent / 현대 동등물 |
|---|---|---|
| Spinning rotor measurement / 스핀 위성 측정 | TI320C10 onboard despin before FFT (24 v/s) | DSCOVR/PlasMag, IMAP/MAG onboard processing |
| Coordinate transforms / 좌표 변환 | GSE→GSM via dipole tilt $\psi$ | Hapgood (1992) full IGRF-based transforms |
| Magnetic-cloud detection / 자기 구름 검출 | Burlaga (1995) flux-rope criteria | Lepping/Lundquist fits, machine-learning ICME catalogues (Nguyen et al. 2019) |
| Real-time alert / 실시간 경보 | 1 v/s RTSW → NOAA SEC ($B_z<-10$ nT, 15 min) | NOAA SWPC G-scale watch + Wing Kp model ingesting ACE/DSCOVR/IMAP |

**Key takeaways implemented in this notebook / 본 노트북에서 구현한 핵심 시사점**:
1. The despin operation is a Z-axis rotation by $-\phi(t)$ that removes the spin tone at $f_s = 1/12$ Hz from $(B_x, B_y)$ while leaving $B_z$ untouched.
   디스핀 연산은 Z축 주변 $-\phi(t)$ 회전으로 $(B_x, B_y)$의 $f_s = 1/12$ Hz 스핀 톤을 제거하지만 $B_z$는 변하지 않는다.
2. GSE → GSM is a single rotation about $\hat{X}$ by the dipole-tilt angle $\psi$; a fixed GSE field acquires a seasonal GSM modulation up to $\pm 35^\circ$.
   GSE → GSM은 쌍극자 기울기 각 $\psi$ 만큼의 $\hat{X}$ 축 회전 하나이며, 고정된 GSE 자기장은 최대 $\pm 35^\circ$의 계절 GSM 변조를 갖는다.
3. A magnetic cloud is recognised by combined criteria: enhanced $|B|$, smooth $B_z$ rotation, and a smoothness score that distinguishes flux ropes from turbulent ICME sheaths.
   자기 구름은 $|B|$ 증가, 부드러운 $B_z$ 회전, 그리고 flux rope를 난류 ICME sheath와 구분하는 smoothness 점수의 결합 기준으로 인식된다.
4. The sustained $B_z < B_\text{th}$ alert is a one-line operational rule that, paired with the L1 → Earth propagation lag, yields the 30–60 min advance warning the modern grid relies on.
   지속된 $B_z < B_\text{th}$ 경보는 L1 → 지구 전파 지연과 결합하여 현대 전력망이 의존하는 30–60분 사전 경보를 만드는 한 줄짜리 운용 규칙이다.